# 05 - Human-in-the-Loop Review

This notebook flags the most uncertain employee-department pairs, lets a person approve, reject, or force a choice, and then writes the review log and final assignment.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
candidate_roots = [current, current.parent, current.parent.parent]
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'smartassign_pipeline.py').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root.')

sys.path.insert(0, str(ROOT / 'src'))
from smartassign_pipeline import (
    MODELS_DIR,
    RESULTS_DIR,
    ModelBundle,
    build_score_and_uncertainty_matrices,
    compute_department_capacities,
    compute_department_profiles,
    ensure_output_directories,
    get_counterfactual_profile_columns,
    load_raw_data,
    save_csv,
    solve_assignment_with_overrides,
    split_train_test,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)

df = load_raw_data()
bundle = ModelBundle.load(MODELS_DIR / 'best_model_bundle.joblib')
train_x, test_x, train_y, test_y = split_train_test(df, bundle.feature_columns)
sample_df = df.loc[test_x.index].copy().sample(n=min(50, len(test_x)), random_state=42)

profile_columns = get_counterfactual_profile_columns(bundle.feature_set_name, df)
department_profiles = compute_department_profiles(df, profile_columns)
score_matrix, uncertainty_matrix = build_score_and_uncertainty_matrices(bundle, sample_df, department_profiles, profile_columns)
capacities = compute_department_capacities(df, len(sample_df))
proposed_assignment, proposed_total = solve_assignment_with_overrides(
    score_matrix=score_matrix,
    employees_ids=sample_df.index,
    projects_ids=score_matrix.columns,
    capacity=capacities,
)

uncertainty_long = uncertainty_matrix.reset_index().melt(id_vars='index', var_name='department', value_name='uncertainty').rename(columns={'index': 'employee_index'})
review_frame = proposed_assignment.merge(uncertainty_long, on=['employee_index', 'department'], how='left')
review_frame = review_frame.sort_values('uncertainty', ascending=False).reset_index(drop=True)
review_count = max(1, int(round(len(review_frame) * 0.15)))
review_queue = review_frame.head(review_count).copy()

save_csv(review_queue, RESULTS_DIR / 'review_queue.csv')
display(Markdown('### Proposed assignment before review'))
display(proposed_assignment.head())
display(Markdown(f'Flagging the top {review_count} of {len(review_frame)} pairs for human review.'))
display(review_queue[['employee_index', 'department', 'predicted_score', 'uncertainty']])
print('Review queue saved to:', RESULTS_DIR / 'review_queue.csv')

class ReviewController:
    def __init__(self, queue_frame, employee_frame, score_frame, capacity_map, output_dir):
        self.queue_frame = queue_frame.reset_index(drop=True)
        self.employee_frame = employee_frame
        self.score_frame = score_frame
        self.capacity_map = capacity_map
        self.output_dir = output_dir
        self.position = 0
        self.locked_pairs = []
        self.forbidden_pairs = []
        self.log_rows = []

        self.decision = widgets.Dropdown(
            options=['Approve', 'Reject - forbid', 'Force - lock'],
            value='Approve',
            description='Decision:',
        )
        self.reason = widgets.Textarea(
            value='',
            placeholder='Optional reason for the decision',
            description='Reason:',
            layout=widgets.Layout(width='100%', height='90px'),
        )
        self.save_button = widgets.Button(description='Save decision', button_style='primary')
        self.finalize_button = widgets.Button(description='Finalize assignment', button_style='success')
        self.status = widgets.Output()
        self.details = widgets.Output()
        self.save_button.on_click(self._on_save)
        self.finalize_button.on_click(self._on_finalize)

    def _current_pair(self):
        if self.position >= len(self.queue_frame):
            return None
        return self.queue_frame.iloc[self.position]

    def _render_details(self):
        self.details.clear_output()
        with self.details:
            if self.position >= len(self.queue_frame):
                display(Markdown('### Review complete\nAll flagged pairs have been processed. Click **Finalize assignment** to write the final files.'))
                return
            row = self._current_pair()
            employee_id = row['employee_index']
            department = row['department']
            employee_row = self.employee_frame.loc[employee_id]
            key_fields = [
                'Department',
                'Job_Title',
                'Gender',
                'Years_At_Company',
                'Performance_Score',
                'Employee_Satisfaction_Score',
                'Training_Hours',
                'Projects_Handled',
                'Promotions',
                'Sick_Days',
            ]
            feature_view = employee_row[[field for field in key_fields if field in employee_row.index]].to_frame('value')
            display(Markdown(f'### Pair {self.position + 1} of {len(self.queue_frame)}'))
            display(Markdown(
                f'**Employee_ID:** {employee_id}\n\n'
                f'**Proposed Department:** {department}\n\n'
                f'**Predicted score:** {float(row["predicted_score"]):.3f}\n\n'
                f'**Uncertainty:** {float(row["uncertainty"]):.3f}'
            ))
            display(feature_view)

    def _write_log(self):
        log_frame = pd.DataFrame(self.log_rows)
        if not log_frame.empty:
            save_csv(log_frame, self.output_dir / 'human_review_log.csv')
        return log_frame

    def _on_save(self, _button):
        with self.status:
            self.status.clear_output()
            row = self._current_pair()
            if row is None:
                print('No more flagged pairs to review.')
                return
            employee_id = row['employee_index']
            department = row['department']
            action = self.decision.value
            reason_text = self.reason.value.strip()
            timestamp = pd.Timestamp.utcnow().isoformat()
            if action == 'Reject - forbid':
                self.forbidden_pairs.append((employee_id, department))
            elif action == 'Force - lock':
                self.locked_pairs.append((employee_id, department))
            self.log_rows.append({
                'timestamp': timestamp,
                'employee_id': employee_id,
                'department': department,
                'action': action,
                'reason': reason_text,
                'predicted_score': float(row['predicted_score']),
                'uncertainty': float(row['uncertainty']),
            })
            self.position += 1
            self.reason.value = ''
            self._render_details()
            self._write_log()
            print(f'Saved decision for employee {employee_id} -> {department}: {action}')

    def _on_finalize(self, _button):
        with self.status:
            self.status.clear_output()
            final_assignment, final_total = solve_assignment_with_overrides(
                score_matrix=self.score_frame,
                employees_ids=self.employee_frame.index,
                projects_ids=self.score_frame.columns,
                capacity=self.capacity_map,
                locked_pairs=self.locked_pairs,
                forbidden_pairs=self.forbidden_pairs,
            )
            self._write_log()
            save_csv(final_assignment, self.output_dir / 'optimal_assignment.csv')
            summary = pd.DataFrame([
                {
                    'sample_size': len(self.queue_frame),
                    'reviewed_pairs': len(self.log_rows),
                    'locked_pairs': len(self.locked_pairs),
                    'forbidden_pairs': len(self.forbidden_pairs),
                    'final_total_predicted_score': final_total,
                }
            ])
            save_csv(summary, self.output_dir / 'human_review_summary.csv')
            display(Markdown('### Final assignment saved'))
            display(summary)
            display(final_assignment.head())

    def show(self):
        self._render_details()
        controls = widgets.VBox([self.decision, self.reason, widgets.HBox([self.save_button, self.finalize_button])])
        display(widgets.VBox([self.details, controls, self.status]))

review_controller = ReviewController(
    queue_frame=review_queue,
    employee_frame=sample_df,
    score_frame=score_matrix,
    capacity_map=capacities,
    output_dir=RESULTS_DIR,
)
review_controller.show()